# E5 GRPO for Track2 Qwen3.5-9B

This notebook continues from the E1 SFT adapter and optimizes responses with Group Relative Policy Optimization. The reward is supplied by the Track1 MoCo-SCL value classifier, so no value head or critic is trained.

In [1]:
import os
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

'false'

In [2]:
from pathlib import Path

ROOT_DIR = Path(os.environ.get("NLPCC_ROOT", "/home/yangdejin/nlpcc/nlpcc_task2"))
DATA_DIR = ROOT_DIR / "data"
TRAIN_DATA_FILE = DATA_DIR / "track2" / "sft_train.jsonl"
DEV_DATA_FILE = DATA_DIR / "track2" / "sft_dev.jsonl"

SFT_MODEL_DIR = ROOT_DIR / "outputs" / "track2" / "qwen35_9b" / "sft"
TRACK1_MODEL_DIR = ROOT_DIR / "outputs" / "track1" / "encoder" / "deberta_v2_xxlarge_moco_scl" / "best_macro_f1_model" / "query_encoder_lora"
OUTPUTS_DIR = ROOT_DIR / "outputs" / "track2" / "qwen35_9b" / "grpo"
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

TRACK1_BASE_MODEL = os.environ.get("TRACK1_BASE_MODEL", "microsoft/deberta-v2-xxlarge")
TRACK2_BASE_MODEL = os.environ.get("TRACK2_BASE_MODEL", "Qwen/Qwen3.5-9B")
PROMPT_MAX_LENGTH = int(os.environ.get("PROMPT_MAX_LENGTH", "512"))
MAX_NEW_TOKENS = int(os.environ.get("MAX_NEW_TOKENS", "64"))
TRACK1_MAX_LENGTH = int(os.environ.get("TRACK1_MAX_LENGTH", "512"))

LOAD_IN_4BIT = os.environ.get("LOAD_IN_4BIT", "1") == "1"
BF16 = os.environ.get("BF16", "1") == "1"
FP16 = os.environ.get("FP16", "0") == "1"
ATTN_IMPLEMENTATION = os.environ.get("ATTN_IMPLEMENTATION", "eager")

GROUP_SIZE = int(os.environ.get("GRPO_GROUP_SIZE", "4"))
PROMPT_BATCH_SIZE = int(os.environ.get("GRPO_PROMPT_BATCH_SIZE", "2"))
MINI_BATCH_SIZE = int(os.environ.get("GRPO_MINI_BATCH_SIZE", "1"))
GRAD_ACCUM_STEPS = int(os.environ.get("GRPO_GRAD_ACCUM_STEPS", "8"))
TRAIN_EPOCHS = int(os.environ.get("GRPO_TRAIN_EPOCHS", "1"))
UPDATE_EPOCHS = int(os.environ.get("GRPO_UPDATE_EPOCHS", "1"))
MAX_STEPS = int(os.environ.get("GRPO_MAX_STEPS", "0"))
LEARNING_RATE = float(os.environ.get("GRPO_LR", "5e-7"))
CLIP_EPS = float(os.environ.get("GRPO_CLIP_EPS", "0.2"))
MAX_GRAD_NORM = float(os.environ.get("GRPO_MAX_GRAD_NORM", "0.3"))
ADV_EPS = float(os.environ.get("GRPO_ADV_EPS", "1e-6"))

GENERATION_BATCH_SIZE = int(os.environ.get("GRPO_GENERATION_BATCH_SIZE", "2"))
TEMPERATURE = float(os.environ.get("GRPO_TEMPERATURE", "0.8"))
TOP_P = float(os.environ.get("GRPO_TOP_P", "0.9"))
SEED = int(os.environ.get("SEED", "42"))

print("root:", ROOT_DIR)
print("sft adapter:", SFT_MODEL_DIR)
print("track1 reward model:", TRACK1_MODEL_DIR)
print("output:", OUTPUTS_DIR)

root: /home/yangdejin/nlpcc/nlpcc_task2
sft adapter: /home/yangdejin/nlpcc/nlpcc_task2/outputs/track2/qwen35_9b/sft
track1 reward model: /home/yangdejin/nlpcc/nlpcc_task2/outputs/track1/encoder/deberta_v2_xxlarge_moco_scl/best_macro_f1_model/query_encoder_lora
output: /home/yangdejin/nlpcc/nlpcc_task2/outputs/track2/qwen35_9b/grpo


In [3]:
import json
import random
import re
import numpy as np
import torch
from tqdm.auto import tqdm

VALUE_LABELS = [
    "Self-direction–thought",
    "Self-direction–action",
    "Stimulation",
    "Hedonism",
    "Achievement",
    "Power–dominance",
    "Power–resources",
    "Face",
    "Security–personal",
    "Security–societal",
    "Tradition",
    "Conformity–rules",
    "Conformity–interpersonal",
    "Humility",
    "Benevolence–dependability",
    "Benevolence–caring",
    "Universalism–concern",
    "Universalism–nature",
    "Universalism–tolerance",
]

VALUE_DEFINITIONS = {
    "Self-direction–thought": "Freedom to cultivate one's own ideas and abilities",
    "Self-direction–action": "Freedom to determine one's own actions",
    "Stimulation": "Excitement, novelty, and change",
    "Hedonism": "Pleasure and sensuous gratification",
    "Achievement": "Success according to social standards",
    "Power–dominance": "Power through exercising control over people",
    "Power–resources": "Power through control of material and social resources",
    "Face": "Security and power through maintaining one's public image and avoiding humiliation",
    "Security–personal": "Safety in one's immediate environment",
    "Security–societal": "Safety and stability in the wider society",
    "Tradition": "Maintaining and preserving cultural, family, or religious traditions",
    "Conformity–rules": "Compliance with rules, laws, and formal obligations",
    "Conformity–interpersonal": "Avoidance of upsetting or harming other people",
    "Humility": "Recognizing one's insignificance in the larger scheme of things",
    "Benevolence–dependability": "Being a reliable and trustworthy member of the ingroup",
    "Benevolence–caring": "Devotion to the welfare of ingroup members",
    "Universalism–concern": "Commitment to equality, justice, and protection for all people",
    "Universalism–nature": "Preservation of the natural environment",
    "Universalism–tolerance": "Acceptance and understanding of those who are different from oneself",
}

label2id = {label: i for i, label in enumerate(VALUE_LABELS)}
id2label = {i: label for i, label in enumerate(VALUE_LABELS)}

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)

def read_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

def normalize_space(text):
    return re.sub(r"\s+", " ", str(text)).strip()

def get_target_value(prompt):
    value = prompt.split("\n\nTarget value:\n", 1)[1]
    value = value.split("\n\nTarget value definition:\n", 1)[0]
    return normalize_space(value)

def build_prompt_from_raw(row):
    value = normalize_space(row["Value"])
    return (
        "You are given a scenario, a question, and a target human value. "
        "Generate one concise, meaningful response that answers the question, fits the scenario, "
        "and naturally aligns with the target value.\n\n"
        f"Scenario:\n{normalize_space(row['Scenario'])}\n\n"
        f"Question:\n{normalize_space(row['Question'])}\n\n"
        f"Target value:\n{value}\n\n"
        f"Target value definition:\n{VALUE_DEFINITIONS[value]}\n\n"
        "Response:\n"
    )

def prepare_rows(path):
    rows = []
    for i, row in enumerate(read_jsonl(path)):
        prompt = row.get("prompt") or build_prompt_from_raw(row)
        target_value = row.get("target_value") or row.get("Value") or get_target_value(prompt)
        rows.append({
            "idx": i,
            "prompt": prompt,
            "target_value": normalize_space(target_value),
        })
    return rows

train_rows = prepare_rows(TRAIN_DATA_FILE)
dev_rows = prepare_rows(DEV_DATA_FILE)
print("train rows:", len(train_rows))
print("dev rows:", len(dev_rows))
print("sample target:", train_rows[0]["target_value"])
print(train_rows[0]["prompt"][:700])

train rows: 3520
dev rows: 514
sample target: Conformity–interpersonal
You are given a scenario, a question, and a target human value. Generate one concise, meaningful response that answers the question, fits the scenario, and naturally aligns with the target value.

Scenario:
You work in a corporate setting where your manager frequently imposes strict guidelines.

Question:
How would you handle a disagreement with a superior during a team meeting?

Target value:
Conformity–interpersonal

Target value definition:
Avoidance of upsetting or harming other people.

Response:



In [4]:
from transformers import AutoTokenizer

tokenizer_source = str(SFT_MODEL_DIR) if SFT_MODEL_DIR.exists() else TRACK2_BASE_MODEL
tokenizer = AutoTokenizer.from_pretrained(tokenizer_source, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

print("tokenizer:", tokenizer_source)
print("pad token:", tokenizer.pad_token, tokenizer.pad_token_id)
print("eos token:", tokenizer.eos_token, tokenizer.eos_token_id)

tokenizer: /home/yangdejin/nlpcc/nlpcc_task2/outputs/track2/qwen35_9b/sft
pad token: <|endoftext|> 248044
eos token: <|im_end|> 248046


In [5]:
from transformers import AutoModel, AutoModelForSequenceClassification, AutoTokenizer
from transformers.modeling_outputs import SequenceClassifierOutput
from peft import PeftModel
import torch.nn as nn

def mean_pooling(last_hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1).type_as(last_hidden_state)
    summed = (last_hidden_state * mask).sum(dim=1)
    counts = mask.sum(dim=1).clamp(min=1e-9)
    return summed / counts

class Track1CompactClassifier(nn.Module):
    def __init__(self, encoder, classifier):
        super().__init__()
        self.encoder = encoder
        self.classifier = classifier

    def forward(self, input_ids=None, attention_mask=None, token_type_ids=None, **kwargs):
        encoder_kwargs = {"input_ids": input_ids, "attention_mask": attention_mask}
        if token_type_ids is not None:
            encoder_kwargs["token_type_ids"] = token_type_ids
        outputs = self.encoder(**encoder_kwargs)
        pooled = mean_pooling(outputs.last_hidden_state, attention_mask)
        logits = self.classifier(pooled.to(self.classifier.weight.dtype))
        return SequenceClassifierOutput(logits=logits)

def torch_load_cpu(path):
    try:
        return torch.load(str(path), map_location="cpu", weights_only=False)
    except TypeError:
        return torch.load(str(path), map_location="cpu")

def load_track1_model(model_dir):
    model_dir = Path(model_dir)
    has_lora_adapter = (model_dir / "adapter_config.json").exists()
    track1_dir = model_dir.parent if has_lora_adapter else model_dir
    track1_tokenizer = AutoTokenizer.from_pretrained(str(track1_dir), use_fast=True, trust_remote_code=True)
    kwargs = dict(
        num_labels=len(VALUE_LABELS),
        id2label=id2label,
        label2id=label2id,
        trust_remote_code=True,
    )
    heads_file = track1_dir / "heads.pt"
    if has_lora_adapter and heads_file.exists():
        heads = torch_load_cpu(heads_file)
        base_model_name = heads.get("model_name", TRACK1_BASE_MODEL)
        if torch.cuda.is_available() and torch.cuda.is_bf16_supported():
            model_dtype = torch.bfloat16
        elif torch.cuda.is_available():
            model_dtype = torch.float16
        else:
            model_dtype = torch.float32
        base_encoder = AutoModel.from_pretrained(
            base_model_name,
            trust_remote_code=True,
            torch_dtype=model_dtype,
        )
        encoder = PeftModel.from_pretrained(base_encoder, str(model_dir))
        hidden_size = getattr(encoder.config, "hidden_size", None)
        if hidden_size is None:
            hidden_size = encoder.base_model.model.config.hidden_size
        classifier = nn.Linear(hidden_size, len(VALUE_LABELS))
        classifier.load_state_dict(heads["classifier"])
        track1_model = Track1CompactClassifier(encoder, classifier)
    elif has_lora_adapter:
        base_model = AutoModelForSequenceClassification.from_pretrained(TRACK1_BASE_MODEL, **kwargs)
        track1_model = PeftModel.from_pretrained(base_model, str(model_dir))
    else:
        track1_model = AutoModelForSequenceClassification.from_pretrained(str(model_dir), **kwargs)

    for p in track1_model.parameters():
        p.requires_grad_(False)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    track1_model.to(device)
    track1_model.eval()
    return track1_model, track1_tokenizer, device

print("loading track1 reward model...")
track1_model, track1_tokenizer, track1_device = load_track1_model(TRACK1_MODEL_DIR)
print("loaded on", track1_device)

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


loading track1 reward model...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/778 [00:00<?, ?it/s]

[transformers] DebertaV2Model LOAD REPORT from: microsoft/deberta-v2-xxlarge
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


loaded on cuda


In [6]:
@torch.no_grad()
def compute_rewards(responses, target_values):
    texts = ["Response: " + normalize_space(response) for response in responses]
    inputs = track1_tokenizer(
        texts,
        max_length=TRACK1_MAX_LENGTH,
        truncation=True,
        padding=True,
        return_tensors="pt",
    )
    inputs = {key: value.to(track1_device) for key, value in inputs.items()}
    outputs = track1_model(**inputs)
    probs = torch.softmax(outputs.logits.float(), dim=-1).cpu().numpy()

    rewards = []
    records = []
    for response, target_value, prob in zip(responses, target_values, probs):
        response = normalize_space(response)
        target_id = label2id[target_value]
        pred_id = int(prob.argmax())
        sorted_ids = np.argsort(prob)[::-1]
        best_other_id = next(i for i in sorted_ids if i != target_id)

        target_prob = float(prob[target_id])
        margin = float(prob[target_id] - prob[best_other_id])
        is_match = float(pred_id == target_id)
        word_count = len(response.split())

        length_penalty = 0.0
        if word_count < 8:
            length_penalty += 0.10
        if word_count > 70:
            length_penalty += 0.05 + 0.002 * (word_count - 70)
        if not response:
            length_penalty += 0.25

        reward = target_prob + 0.25 * margin + 0.10 * is_match - length_penalty
        rewards.append(float(reward))
        records.append({
            "target_value": target_value,
            "pred_value": id2label[pred_id],
            "response": response,
            "target_prob": target_prob,
            "margin": margin,
            "word_count": word_count,
            "reward": float(reward),
        })
    return np.asarray(rewards, dtype=np.float32), records

In [7]:
import gc
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel, prepare_model_for_kbit_training

def model_init_kwargs():
    compute_dtype = torch.bfloat16 if BF16 else torch.float16
    kwargs = {
        "trust_remote_code": True,
        "low_cpu_mem_usage": True,
        "attn_implementation": ATTN_IMPLEMENTATION,
    }
    if LOAD_IN_4BIT:
        kwargs["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=compute_dtype,
            bnb_4bit_use_double_quant=True,
        )
        kwargs["device_map"] = "auto"
    else:
        kwargs["torch_dtype"] = compute_dtype
        kwargs["device_map"] = "auto" if torch.cuda.is_available() else None
    return kwargs

def load_policy_model():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    base_model = AutoModelForCausalLM.from_pretrained(TRACK2_BASE_MODEL, **model_init_kwargs())
    if base_model.get_input_embeddings().num_embeddings != len(tokenizer):
        base_model.resize_token_embeddings(len(tokenizer))
    base_model.config.pad_token_id = tokenizer.pad_token_id
    base_model.config.use_cache = False
    if LOAD_IN_4BIT:
        base_model = prepare_model_for_kbit_training(base_model)

    model = PeftModel.from_pretrained(base_model, str(SFT_MODEL_DIR), is_trainable=True)
    model.config.use_cache = False
    model.gradient_checkpointing_enable()
    if hasattr(model, "print_trainable_parameters"):
        model.print_trainable_parameters()
    return model

policy_model = load_policy_model()
policy_device = next(policy_model.parameters()).device
print("policy device:", policy_device)

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/427 [00:00<?, ?it/s]

trainable params: 29,097,984 || all params: 8,980,910,592 || trainable%: 0.3240
policy device: cuda:0


## GRPO rollout and loss

For each prompt, sample `GROUP_SIZE` responses, compute Track1 rewards, normalize rewards inside the prompt group, and update the policy with a clipped importance-ratio objective.

In [8]:
def iter_wrapped_models(model):
    seen = set()
    stack = [model]
    while stack:
        obj = stack.pop()
        if obj is None or id(obj) in seen:
            continue
        seen.add(id(obj))
        yield obj
        for attr in ["module", "pretrained_model", "base_model", "model"]:
            stack.append(getattr(obj, attr, None))

def set_gradient_checkpointing(model, enabled):
    method_name = "gradient_checkpointing_enable" if enabled else "gradient_checkpointing_disable"
    for obj in iter_wrapped_models(model):
        method = getattr(obj, method_name, None)
        if callable(method):
            try:
                method()
            except TypeError:
                pass

def set_use_cache(model, enabled):
    for obj in iter_wrapped_models(model):
        config = getattr(obj, "config", None)
        if config is not None and hasattr(config, "use_cache"):
            config.use_cache = enabled

def encode_prompts(rows):
    return tokenizer(
        [row["prompt"] for row in rows],
        max_length=PROMPT_MAX_LENGTH,
        truncation=True,
        padding=True,
        return_tensors="pt",
    )

def make_labels(full_ids, prompt_width):
    labels = full_ids.clone()
    labels[:, :prompt_width] = -100
    labels = labels.masked_fill(labels == tokenizer.pad_token_id, -100)
    return labels

def pad_generated_chunks(chunks):
    max_len = max(chunk.shape[1] for chunk in chunks)
    padded = []
    for chunk in chunks:
        if chunk.shape[1] == max_len:
            padded.append(chunk)
            continue
        pad_width = max_len - chunk.shape[1]
        pad = torch.full(
            (chunk.shape[0], pad_width),
            tokenizer.pad_token_id,
            dtype=chunk.dtype,
            device=chunk.device,
        )
        padded.append(torch.cat([chunk, pad], dim=1))
    return torch.cat(padded, dim=0)

def token_logps(model, input_ids, attention_mask, labels):
    outputs = model(input_ids=input_ids, attention_mask=attention_mask)
    logits = outputs.logits[:, :-1, :].float()
    shift_labels = labels[:, 1:].clone()
    mask = shift_labels != -100
    safe_labels = shift_labels.masked_fill(~mask, 0)
    log_probs = logits.log_softmax(dim=-1)
    selected = torch.gather(log_probs, dim=2, index=safe_labels.unsqueeze(2)).squeeze(2)
    selected = selected * mask
    lengths = mask.sum(dim=1).clamp(min=1)
    sequence_logps = selected.sum(dim=1)
    average_logps = sequence_logps / lengths
    return selected, mask, sequence_logps, average_logps, lengths

@torch.no_grad()
def generate_rollout(rows):
    policy_model.eval()
    set_gradient_checkpointing(policy_model, False)
    set_use_cache(policy_model, True)
    encoded = encode_prompts(rows)
    input_ids = encoded["input_ids"].to(policy_device)
    attention_mask = encoded["attention_mask"].to(policy_device)
    prompt_width = input_ids.shape[1]

    repeated_input_ids = input_ids.repeat_interleave(GROUP_SIZE, dim=0)
    repeated_attention_mask = attention_mask.repeat_interleave(GROUP_SIZE, dim=0)
    repeated_targets = [row["target_value"] for row in rows for _ in range(GROUP_SIZE)]
    group_ids = np.repeat(np.arange(len(rows)), GROUP_SIZE)

    generated_chunks = []
    for start in range(0, repeated_input_ids.shape[0], GENERATION_BATCH_SIZE):
        end = start + GENERATION_BATCH_SIZE
        generated = policy_model.generate(
            input_ids=repeated_input_ids[start:end],
            attention_mask=repeated_attention_mask[start:end],
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=True,
            temperature=TEMPERATURE,
            top_p=TOP_P,
            use_cache=True,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
        generated_chunks.append(generated)
    full_ids = pad_generated_chunks(generated_chunks)
    response_ids = full_ids[:, prompt_width:]
    full_attention_mask = full_ids.ne(tokenizer.pad_token_id).long()
    labels = make_labels(full_ids, prompt_width)
    responses = tokenizer.batch_decode(response_ids, skip_special_tokens=True)
    responses = [normalize_space(response) for response in responses]

    old_token_logps, old_loss_mask, _, old_avg_logps, lengths = token_logps(
        policy_model,
        full_ids,
        full_attention_mask,
        labels,
    )
    rewards, reward_records = compute_rewards(responses, repeated_targets)

    set_use_cache(policy_model, False)
    set_gradient_checkpointing(policy_model, True)
    policy_model.train()

    samples = []
    for i in range(full_ids.shape[0]):
        samples.append({
            "input_ids": full_ids[i].detach().cpu(),
            "attention_mask": full_attention_mask[i].detach().cpu(),
            "labels": labels[i].detach().cpu(),
            "old_token_logps": old_token_logps[i].detach().cpu(),
            "old_loss_mask": old_loss_mask[i].detach().cpu(),
            "old_avg_logp": float(old_avg_logps[i].detach().cpu()),
            "length": int(lengths[i].detach().cpu()),
            "reward": float(rewards[i]),
            "group_id": int(group_ids[i]),
            "target_value": repeated_targets[i],
            "response": responses[i],
            "record": reward_records[i],
        })
    return samples

def add_group_advantages(samples, min_std=None):
    kept = []
    groups = sorted({sample["group_id"] for sample in samples})
    for group_id in groups:
        group = [sample for sample in samples if sample["group_id"] == group_id]
        rewards = np.asarray([sample["reward"] for sample in group], dtype=np.float32)
        std = float(rewards.std())
        if min_std is not None and std < min_std:
            continue
        mean = float(rewards.mean())
        denom = std + ADV_EPS
        for sample in group:
            item = dict(sample)
            item["advantage"] = float((sample["reward"] - mean) / denom) if std > 0 else 0.0
            item["group_reward_mean"] = mean
            item["group_reward_std"] = std
            kept.append(item)
    return kept

def pad_1d(values, pad_value, dtype):
    max_len = max(value.numel() for value in values)
    output = torch.full((len(values), max_len), pad_value, dtype=dtype)
    for i, value in enumerate(values):
        output[i, :value.numel()] = value.to(dtype=dtype)
    return output

def collate_rollout(samples):
    batch = {
        "input_ids": pad_1d([sample["input_ids"] for sample in samples], tokenizer.pad_token_id, torch.long),
        "attention_mask": pad_1d([sample["attention_mask"] for sample in samples], 0, torch.long),
        "labels": pad_1d([sample["labels"] for sample in samples], -100, torch.long),
        "old_token_logps": pad_1d([sample["old_token_logps"] for sample in samples], 0.0, torch.float32),
        "old_loss_mask": pad_1d([sample["old_loss_mask"] for sample in samples], False, torch.bool),
        "old_avg_logp": torch.tensor([sample["old_avg_logp"] for sample in samples], dtype=torch.float32),
        "advantage": torch.tensor([sample["advantage"] for sample in samples], dtype=torch.float32),
    }
    return {key: value.to(policy_device) for key, value in batch.items()}

def shuffled_batches(rows, batch_size):
    indices = list(range(len(rows)))
    random.shuffle(indices)
    for start in range(0, len(indices), batch_size):
        yield [rows[i] for i in indices[start:start + batch_size]]

def optimizer_step_if_ready(loss, step_in_accum, optimizer):
    loss.backward()
    if step_in_accum % GRAD_ACCUM_STEPS == 0:
        torch.nn.utils.clip_grad_norm_(policy_model.parameters(), MAX_GRAD_NORM)
        optimizer.step()
        optimizer.zero_grad(set_to_none=True)
        return True
    return False

In [9]:
def grpo_loss(batch):
    _, _, _, new_avg_logps, _ = token_logps(
        policy_model,
        batch["input_ids"],
        batch["attention_mask"],
        batch["labels"],
    )
    old_avg_logps = batch["old_avg_logp"]
    advantages = batch["advantage"]

    log_ratio = (new_avg_logps - old_avg_logps).clamp(min=-20, max=20)
    ratio = torch.exp(log_ratio)
    clipped_ratio = torch.clamp(ratio, 1.0 - CLIP_EPS, 1.0 + CLIP_EPS)
    policy_loss = -torch.minimum(ratio * advantages, clipped_ratio * advantages).mean()

    approx_kl = (old_avg_logps - new_avg_logps).mean().detach()
    clip_fraction = (torch.abs(ratio - 1.0) > CLIP_EPS).float().mean().detach()
    return policy_loss, {
        "loss": float(policy_loss.detach().cpu()),
        "approx_kl": float(approx_kl.cpu()),
        "clip_fraction": float(clip_fraction.cpu()),
        "ratio_mean": float(ratio.detach().mean().cpu()),
    }

In [10]:
from torch.optim import AdamW

optimizer = AdamW((p for p in policy_model.parameters() if p.requires_grad), lr=LEARNING_RATE)
optimizer.zero_grad(set_to_none=True)

update_count = 0
micro_step = 0
stop_training = False

for epoch in range(TRAIN_EPOCHS):
    progress = tqdm(shuffled_batches(train_rows, PROMPT_BATCH_SIZE), total=(len(train_rows) + PROMPT_BATCH_SIZE - 1) // PROMPT_BATCH_SIZE, desc=f"grpo epoch {epoch + 1}")
    for rows in progress:
        rollout = generate_rollout(rows)
        rollout = add_group_advantages(rollout)
        if not rollout:
            continue

        rewards = [sample["reward"] for sample in rollout]
        advantages = [sample["advantage"] for sample in rollout]
        random.shuffle(rollout)

        for _ in range(UPDATE_EPOCHS):
            for start in range(0, len(rollout), MINI_BATCH_SIZE):
                mini_samples = rollout[start:start + MINI_BATCH_SIZE]
                batch = collate_rollout(mini_samples)
                loss, metrics = grpo_loss(batch)
                micro_step += 1
                stepped = optimizer_step_if_ready(loss / GRAD_ACCUM_STEPS, micro_step, optimizer)
                if stepped:
                    update_count += 1

        progress.set_postfix({
            "reward": f"{np.mean(rewards):.4f}",
            "adv": f"{np.mean(np.abs(advantages)):.3f}",
            "updates": update_count,
        })
        if update_count % 10 == 0:
            best = max(rollout, key=lambda sample: sample["reward"])
            print("=" * 80)
            print("updates:", update_count, "mean reward:", round(float(np.mean(rewards)), 4), "mean abs advantage:", round(float(np.mean(np.abs(advantages))), 4))
            print("target:", best["target_value"], "pred:", best["record"]["pred_value"], "target_prob:", round(best["record"]["target_prob"], 4))
            print("response:", best["response"])
        if MAX_STEPS > 0 and update_count >= MAX_STEPS:
            stop_training = True
            break
    if stop_training:
        break

if micro_step % GRAD_ACCUM_STEPS != 0:
    torch.nn.utils.clip_grad_norm_(policy_model.parameters(), MAX_GRAD_NORM)
    optimizer.step()
    optimizer.zero_grad(set_to_none=True)

policy_model.save_pretrained(str(OUTPUTS_DIR))
tokenizer.save_pretrained(str(OUTPUTS_DIR))
print("saved GRPO adapter to:", OUTPUTS_DIR)

grpo epoch 1:   0%|          | 0/1760 [00:00<?, ?it/s]

updates: 10 mean reward: 1.3244 mean abs advantage: 0.8996
target: Achievement pred: Achievement target_prob: 0.9997
response: Focus on maximizing the team's success by dedicating time to refine the presentation, as achieving excellence aligns with professional recognition and long-term career growth.
updates: 20 mean reward: 1.3493 mean abs advantage: 0.8295
target: Security–personal pred: Security–personal target_prob: 0.9998
response: I would immediately decline the offer and insist on leaving alone, firmly prioritizing my personal safety and avoiding any potential danger.
updates: 30 mean reward: 1.3473 mean abs advantage: 0.8646
target: Tradition pred: Tradition target_prob: 0.9996
response: I will insist on using paper offerings, arguing that maintaining our ancestral worship practices is essential for preserving our family's spiritual heritage.
updates: 40 mean reward: 1.3491 mean abs advantage: 0.8812
target: Universalism–nature pred: Universalism–nature target_prob: 0.9997
res

/home/yangdejin/miniconda3/envs/nlpcc/lib/python3.12/site-packages/peft/utils/save_and_load.py:386: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(


saved GRPO adapter to: /home/yangdejin/nlpcc/nlpcc_task2/outputs/track2/qwen35_9b/grpo


In [11]:
def generate_response(prompt, max_new_tokens=96):
    policy_model.eval()
    encoded = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=PROMPT_MAX_LENGTH).to(policy_device)
    with torch.no_grad():
        output = policy_model.generate(
            **encoded,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    response_ids = output[0, encoded["input_ids"].shape[1]:]
    return normalize_space(tokenizer.decode(response_ids, skip_special_tokens=True))

sample = dev_rows[0]
print("target:", sample["target_value"])
print(generate_response(sample["prompt"]))

target: Conformity–interpersonal
I would agree to provide regular updates to maintain harmony and avoid any potential conflict, prioritizing the teammate’s emotional comfort and our smooth collaboration.
